In [ ]:
import h5py
import pandas as pd
from PIL import Image
import numpy as np
import os
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.rich import tqdm

In [ ]:
def create_h5_file(json_data, image_path, output_path):
    image = Image.open(image_path)
    image_array = np.array(image)
    
    with h5py.File(output_path, 'w') as h5f:
        h5f.create_dataset('STM_image', data=image_array, compression="gzip", compression_opts=9)
        
        json_group = h5f.create_group('properties')
        
        for key, value in json_data.items():
            if isinstance(value, (int, float, str, bytes)):
                json_group.attrs[key] = value
            elif isinstance(value, list):
                json_group.create_dataset(key, data=np.array(value))
            else:
                raise ValueError(f"Unsupported data type for key {key}: {type(value)}")

csv_file = './dataset.csv'
image_folder = '/home/mario/jobs/git_folders/AutoDFTB/test_batch/transport/stm_images'
output_folder = './h5_files'

df = pd.read_csv(csv_file)

for index, row in df.iterrows():
    file_name = row[0]
    json_data = row[1:].to_dict()
    print(json_data)
    image_path = os.path.join(image_folder, file_name + '_e.png') 
    output_path = os.path.join(output_folder, file_name + '.h5')
    
    create_h5_file(json_data, image_path, output_path)
    print(f'Creato file H5 per {file_name}')
    


In [ ]:

def print_h5_contents(file_path):
    with h5py.File(file_path, 'r') as h5f:
        print("=== Metadati globali (file-level attrs) ===")
        for key, val in h5f.attrs.items():
            print(f'{key}: {val}')
        print()

        def print_attrs(name, obj):
            if isinstance(obj, h5py.Group):
                print(f'Group: {name}')
                for key, val in obj.attrs.items():
                    print(f'    Attribute: {key} = {val}')
            elif isinstance(obj, h5py.Dataset):
                print(f'Dataset: {name}')
                print(f'    Shape: {obj.shape}')
                print(f'    Dtype: {obj.dtype}')
                for key, val in obj.attrs.items():
                    print(f'    Attribute: {key} = {val}')


        print("\n=== Contenuto del file ===")
        h5f.visititems(print_attrs)


file_path = '/home/mario/jobs/git_folders/AutoDFTB/paper/h5_files/dataset_0.h5'
print_h5_contents(file_path)

In [ ]:
def extract_and_plot_image(file_path, image_dataset_path):
    with h5py.File(file_path, 'r') as h5f:
        if image_dataset_path in h5f:
            image_data = np.array(h5f[image_dataset_path])
            
            plt.imshow(image_data, cmap='gray')
            plt.title('Image from HDF5')
            plt.axis('off')  
            plt.show()
        else:
            print(f"Dataset {image_dataset_path} not found in file {file_path}")

file_path = '/home/mario/jobs/git_folders/AutoDFTB/paper/h5_files/graphene_67.h5'
image_dataset_path = 'STM_image'  
extract_and_plot_image(file_path, image_dataset_path)

In [ ]:
def read_from_xyz_file(file_path: Path):
    """Read xyz files and return lists of x,y,z coordinates and atoms"""
    
    symbol_to_atomic_number = {
        "H": 1, "He": 2, "Li": 3, "Be": 4, "B": 5,
        "C": 6, "N": 7, "O": 8, "F": 9, "Ne": 10,
        "Na": 11, "Mg": 12, "Al": 13, "Si": 14, "P": 15,
        "S": 16, "Cl": 17, "Ar": 18, "K": 19, "Ca": 20,
        # Aggiungi altri elementi se necessario
    }

    X = []
    Y = []
    Z = []
    atoms = []

    with open(str(file_path), "r") as f:
        num_atom = int(next(f))
        next(f)  # ignore the comment

        for _ in range(num_atom):
            l = next(f).split()
            if len(l) == 4 or len(l) == 5:
                X.append(float(l[1]))
                Y.append(float(l[2]))
                Z.append(float(l[3]))
                
                atomic_number = symbol_to_atomic_number.get(str(l[0]))
                if atomic_number is None:
                    raise ValueError(f"Simbolo chimico sconosciuto: {str(l[0])}")
                atoms.append(atomic_number)

    X = np.asarray(X)
    Y = np.asarray(Y)
    Z = np.asarray(Z)
    
    coords = np.stack((X,Y,Z),axis=1)

    return atoms, coords

In [ ]:
#file_name,num_atoms,total_energy,num_electrons,Fermi_energy,electron_affinity,ionization_potential,band_gap,total_energy+1,total_energy-1,current
package_path = Path("/home/tommaso/git_workspace/AutoDFTB")
output_path = Path("./h5_files")
csv_file = Path("./dataset.csv")
samples_for_file = 10000

# Leggi il file CSV
df = pd.read_csv(csv_file)

# Calcola il numero totale di slice necessari
num_slices = len(df) // samples_for_file + (len(df) % samples_for_file != 0)

# Divide il DataFrame in slice
slices = [df[i*samples_for_file:(i+1)*samples_for_file] for i in range(num_slices)]

# Crea un file HDF5
for i, slice_df in enumerate(slices):
    with h5py.File(output_path.joinpath(f"dataset_{i}.h5"), 'w') as h5f:
        # Attributi generali
        h5f.attrs['title'] = 'Dataset Esempio FAIR'
        h5f.attrs['description'] = 'Un esempio di come creare un dataset HDF5 seguendo i principi FAIR.'
        h5f.attrs['author'] = 'Nome Ricercatore'
        h5f.attrs['created'] = '2024-07-23'
        h5f.attrs['license'] = 'CC-BY-4.0'
        h5f.attrs['doi'] = '10.1234/example.doi'
        h5f.attrs['url'] = 'https://zenodo.org/record/1234567'
        h5f.attrs['version'] = '1.0'
        h5f.attrs['last_updated'] = '2024-07-23'
        
        # Documentazione
        docs_group = h5f.create_group('documentation')
        docs_group.attrs['description'] = 'Documentazione completa sul dataset e sulle metodologie utilizzate.'
        docs_group.create_dataset('full_description', data=np.string_(
            "Questo dataset contiene misurazioni di temperatura e pressione raccolte nel corso di un esperimento..."
        ))

        
        pbar = tqdm(total=len(slice_df))
        for index, row in slice_df.iterrows():
            file_name = row[0]
            json_data = row[1:].to_dict()
            image_path = package_path.joinpath(f"data_batch_{json_data['batch']}","transport","stm_images",f"{file_name}_e.png")
            
            # Carica l'immagine e convertila in un array numpy
            image = Image.open(image_path)
            image_array = np.array(image)
            
            master_group = h5f.create_group(f'{file_name}')
            master_group.attrs['description'] = f"Simulated data regarding the flake: {file_name}"
            
            #STM image
            stm_data = master_group.create_dataset('STM_image', data=image_array, compression="gzip", compression_opts=9)
            stm_data.attrs['description'] = "STM image of the flake"
            
            #Optimized geometry
            atoms,coords = read_from_xyz_file(package_path.joinpath(f"data_batch_{json_data['batch']}","dftb","xyz_files_fixed_opt",f"{file_name}_opt.xyz"))
            optimized_geom = master_group.create_group("OptGeom")
            
            atoms_data = optimized_geom.create_dataset("AtomicNumbers",data=atoms)
            atoms_data.attrs['description']=""
            
            coords_data = optimized_geom.create_dataset("Coordinates",data=coords)
            coords_data.attrs['units'] = 'Armstrong'
            coords_data.attrs['description']=""
            
            #Num atoms
            n_atoms_data = master_group.create_dataset("NumAtoms", data=json_data['num_atoms'])
            n_atoms_data.attrs['description'] = "Number of atoms of the flake"
            
            #Num electrons
            n_electrons_data = master_group.create_dataset("NumElectrons", data=json_data['num_electrons'])
            n_electrons_data.attrs['description'] = "Number of electrons of the flake"
            
            #Total energy
            total_energy_data = master_group.create_dataset("TotalEnergy", data=json_data["total_energy"])
            total_energy_data.attrs['units'] = "eV"
            total_energy_data.attrs['description'] = "Total energy of the flake"
            
            #Fermi_energy
            fermi_energy_data = master_group.create_dataset("FermiEnergy", data=json_data["Fermi_energy"])
            fermi_energy_data.attrs['units'] = "eV"
            fermi_energy_data.attrs['description'] = "Fermi energy of the flake"
            
            #EA
            EA_data = master_group.create_dataset("EA", data=json_data["electron_affinity"])
            EA_data.attrs['units'] = "eV"
            EA_data.attrs['description'] = "Electron affinity of the flake"
            
            #IP
            IP_data = master_group.create_dataset("IP", data=json_data["ionization_potential"])
            IP_data.attrs['units'] = "eV"
            IP_data.attrs['description'] = "Ionization potential of the flake"
            
            #Band gap
            band_gap_data = master_group.create_dataset("BandGap", data=json_data["band_gap"])
            band_gap_data.attrs['units'] = "eV"
            band_gap_data.attrs['description'] = "Band gap of the flake"
            
            #Optimized electrodes
            atoms,coords = read_from_xyz_file(package_path.joinpath(f"data_batch_{json_data['batch']}","transport","electrodes",f"{file_name}_e.xyz"))
            electrodes = master_group.create_group("Electode")
            
            e_atoms_data = electrodes.create_dataset("AtomicNumbers",data=atoms)
            e_atoms_data.attrs['description']=""
            
            e_coords_data = electrodes.create_dataset("Coordinates",data=coords)
            e_coords_data.attrs['units'] = 'Armstrong'
            e_coords_data.attrs['description']=""
            
            #Current
            current_data = master_group.create_dataset("TransportCurrent", data=json_data["current"])
            current_data.attrs['units'] = "uA"
            current_data.attrs['description'] = "Transport current of the flake"
            
            pbar.update(1)
            
        pbar.close()

In [ ]:
def print_h5_contents(file_path):
    with h5py.File(file_path, 'r') as h5f:
        def print_attrs(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f'Dataset: {name}')
                print(f'    Shape: {obj.shape}')
                print(f'    Dtype: {obj.dtype}')
                print(f'    Data: {np.array(obj)}\n')
            elif isinstance(obj, h5py.Group):
                print(f'Group: {name}')
                for key, val in obj.attrs.items():
                    print(f'    Attribute: {key} = {val}')
            if isinstance(obj, h5py.AttributeManager):
                for key, val in obj.attrs.items():
                    print(f'    Attribute: {key} = {val}')
        
        h5f.visititems(print_attrs)

# Esempio di utilizzo
file_path = '/home/mario/Mario/Phd_code/AutoDFTB/dataset_0.h5'
print_h5_contents(file_path)

In [ ]:
import h5py
import numpy as np

def print_h5_contents(file_path):
    with h5py.File(file_path, 'r') as h5f:
        print("=== File-level attributes ===")
        for key, val in h5f.attrs.items():
            print(f'  {key}: {val}')
        print()

        def print_attrs(name, obj):
            if isinstance(obj, h5py.Group):
                print(f'Group: {name}')
                for key, val in obj.attrs.items():
                    print(f'    Attribute: {key} = {val}')
                print()
            elif isinstance(obj, h5py.Dataset):
                print(f'Dataset: {name}')
                print(f'    Shape: {obj.shape}')
                print(f'    Dtype: {obj.dtype}')
                for key, val in obj.attrs.items():
                    print(f'    Attribute: {key} = {val}')
                print(f'    Sample data: {np.array(obj)}...')  # solo primi 3 valori
                print()

        h5f.visititems(print_attrs)

# Esempio di utilizzo
file_path = '/home/mario/Mario/Phd_code/AutoDFTB/dataset/h5_files/dataset_0.h5'
print_h5_contents(file_path)


In [5]:
import h5py
import numpy as np 
from pathlib import Path
import json

data_path = Path("/home/mario/Mario/Phd_code/AutoDFTB/dataset/h5_files")

# Metadati comuni
keywords = np.array(["Graphene", "Artificial Intelligence","Density Function Theory", "Digital Technology"], dtype='S')
contacts = np.array(["mario.vozza@polito.it", "francesco.mercuri@cnr.it"], dtype='S')
authors = np.array([
    "Tommaso Forni",
    "Mario Vozza",
    "Alessandro Pecchia",
    "Francesco Mercuri"
], dtype='S')

related_identifiers = [
    {
        "type": "DOI",
        "relation": "isDescribedBy",
        "identifier": "10.1038/s41565-024-01234"
    }
]

funding = (
    "This research was funded by the European Union – NextGenerationEU, through the Italian Ministry of Environment and "
    "Energy Security (POR H2 AdP MMES/ENEA), with the involvement of CNR and RSE. "
    "It was supported by PNRR - Mission 2, Component 2, Investment 3.5 'Ricerca e sviluppo sull’idrogeno' "
    "(CUP: B93C22000630006), and by Mission 04, Component 2, Investment 1.5 – NextGenerationEU "
    "(Call for tender No. 3277 dated 30/12/2021). "
    "This work also received funding from the European Union’s Horizon Europe research and innovation programme "
    "under Grant Agreements No. 101091464 and 101137809. "
    "Additionally, it is based on work from COST Action EuMINe – European Materials Informatics Network (CA22143), "
    "supported by COST (European Cooperation in Science and Technology)."
)

description = (
    "This dataset provides detailed information on simulated defected graphene properties, "
    "following FAIR principles."
)

file_list = list(data_path.glob("*.h5"))
for file in file_list:
    with h5py.File(file, 'r+') as f:
        print(f"Updating: {file.name}")
        
        f.attrs['doi'] = '10.5281/zenodo.13760108'
        f.attrs['last_updated'] = '2025-05-15'
        f.attrs['version'] = '1.0.1'
        f.attrs['url'] = 'https://zenodo.org/records/13760109'
        f.attrs['language'] = 'en'
        f.attrs['title'] = 'A FAIR-Compliant Dataset of Simulated STM, Electronic and Transport Properties of Defected Graphene'
        f.attrs['funding'] = funding
        f.attrs['description'] = description
        f.attrs['created'] = '2024-07-23'
        f.attrs['authors'] = authors
        f.attrs['contact'] = contacts
        f.attrs['keywords'] = keywords
        f.attrs['license'] = 'CC-BY-4.0'
        f.attrs['related_identifiers'] = json.dumps(related_identifiers).encode('utf-8')
        f.attrs['format'] = 'HDF5'
        f.attrs['mime_type'] = 'application/x-hdf'
        f.attrs['generated_by'] = 'DFTB+ 22.1'

# encode as bytes


Updating: dataset_0.h5
Updating: dataset_1.h5
Updating: dataset_2.h5
Updating: dataset_3.h5
Updating: dataset_4.h5
Updating: dataset_5.h5


In [6]:
import h5py

output_path = "root_metadata.txt"
file_path = "/home/mario/Mario/Phd_code/AutoDFTB/dataset/h5_files/dataset_0.h5"
with h5py.File(file_path, "r") as f, open(output_path, "w") as out:
    out.write(f"File: {file_path}\n")
    out.write("Root-level metadata and contents\n")
    out.write("="*50 + "\n\n")

    # Scrive gli attributi (metadati) del root "/"
    out.write("Attributes at root level:\n")
    for key, val in f.attrs.items():
        out.write(f"  - {key} = {val}\n")

    # Scrive gli oggetti presenti nel root (dataset o gruppi)
    out.write("\nObjects at root level:\n")
    for key in f.keys():
        obj = f[key]
        obj_type = "Group" if isinstance(obj, h5py.Group) else "Dataset"
        out.write(f"  - {key} ({obj_type})\n")